## BOOKING SYSTEM

Objective: Control WHO and HOW the net is used on an AP. Not to block the AP physically but to somehow book each class.

1. We need a specific SSID, to create a temporal wifi connection like --> EXAM_CLASSROOM_Q21003

2. Acces control. We need the list of the MAC addresses of the students and their credentials, so that they are the only ones connecting.

3. Manage the politics of the new net --> limit velocity, block access to some pages and priorise the trafic. 

We will simulate the reservations with data like this:

In [2]:
resetvations = [
    {
        "ap": "AP-ETSE69",
        "start": "2025-05-01 10:00",
        "end": "2025-05-01 12:00",
        "num_students": 30
    }
]

PREDICTOR: to know if an AP is good or not we already have a recommender and predictor system. So that we can use a recommender to know wether the number of students is manegable for this AP.

#### MAIN IDEA 
**Teacher says:** I need room Q4/1007 for 30 students on 01/05/2025 between 10:00-12:00


**System returns:**

                You can use AP-ETSE69, it performance is predicted to be good.
or

                You better not use AP-ETSE69, it performance is predicted to be mid, so better change of room.


So we need:

1. Recommender of the AP --> already made. GIVE US WHICH AP TO USE GIVEN A NODE IN THE GRAPH.

2. Prediction of the charge --> already made. It gives us how many clients/performance while using it and the probability that it fails. SO MAYBE IS BETTER TO CHANGE OF ROOM.

Expected output

        AP-ETSE69:
        - SNR esperado ↓
        - carga ↑
        - riesgo: MEDIO

#### SUMMARY
1. Ask reserve
2. Predict the charge
3. Evaluate the near AP
4. Recommend the better one or to move to another room.

Input:
- Room: Q4/1007
- Students: 40
- Start: 01/05/2025 10:00
- End: 01/05/2025 12:00

Output:
- AP recommended: AP-ESTSE69
- Performance: 80%
- Risk of down: 5%
- Alternative: AP-ESTSE80

### 1. BOOKING MANAGEMENT

In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np

In [4]:
df_clean = pd.read_parquet(r"C:\Escriptori\SYNTESIS_PROJECT\results\merged_clean.parquet")

In [31]:
df_clean

,associated_device_name,band,health,speed,maxspeed,signal_db,signal_strength,snr,last_connection_time,snapshot_ts,...,hour,date,signal_integrity,signal_score,performance,swarm_name,cpu_utilization,mem_usage,client_count,overloaded
0,AP-CEDU26,5.0,100.0,65.0,192.0,-56.0,4.0,40.0,1.743588e+12,2025-04-02 22:01:15,...,22,2025-04-02,Excellent+,0.975000,Poor,AP-CEDU26,2.0,60.073740,1.0,False
1,AP-CCOM19,5.0,97.0,6.0,192.0,-71.0,3.0,24.0,NaN,2025-04-02 22:01:15,...,22,2025-04-02,Good,0.830500,Critical,AP-CCOM19,5.0,71.413457,3.0,False
2,AP-ECON06,5.0,100.0,6.0,192.0,-85.0,2.0,11.0,1.743611e+12,2025-04-02 22:01:15,...,22,2025-04-02,Fair,0.632500,Critical,AP-ECON06,5.0,71.506719,2.0,False
3,AP-FTI17,5.0,100.0,6.0,192.0,-85.0,2.0,11.0,1.743588e+12,2025-04-02 22:01:15,...,22,2025-04-02,Fair,0.632500,Critical,AP-FTI17,5.0,70.858237,2.0,False
4,AP-EST08,5.0,100.0,6.0,192.0,-89.0,1.0,7.0,1.743598e+12,2025-04-02 22:01:15,...,22,2025-04-02,Fair,0.569167,Critical,AP-EST08,4.0,67.621338,1.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11919084,AP-ETSE57,5.0,100.0,6.0,173.0,-87.0,1.0,8.0,1.751505e+12,2025-07-10 19:00:27,...,19,2025-07-10,Fair,0.589167,Critical,AP-ETSE57,8.0,67.446683,3.0,False
11919085,AP-MED49,5.0,100.0,6.0,192.0,-69.0,3.0,27.0,1.750155e+12,2025-07-10 19:00:27,...,19,2025-07-10,Good,0.885833,Critical,AP-MED49,12.0,65.185018,32.0,False
11919086,AP-CIVIC12,5.0,100.0,6.0,192.0,-84.0,2.0,11.0,1.750155e+12,2025-07-10 19:00:27,...,19,2025-07-10,Fair,0.636667,Critical,AP-CIVIC12,9.0,69.158311,19.0,False
11919087,AP-MED01,5.0,100.0,6.0,96.0,-85.0,2.0,11.0,1.750155e+12,2025-07-10 19:00:27,...,19,2025-07-10,Fair,0.632500,Critical,AP-MED01,21.0,70.940632,9.0,False


In [5]:
geo = gpd.read_file(r'C:\Escriptori\SYNTESIS_PROJECT\GEOLOCALIZATION\geolocation_package\data\aps_geolocalizados_wgs84.geojson')
room_to_ap = (
    geo[['USER_Espai', 'USER_NOM_A', 'USER_EDIFI', 'Num_Planta']]
    .dropna(subset=['USER_Espai', 'USER_NOM_A'])
    .set_index('USER_Espai')
)

In [6]:
print(geo[['USER_Espai', 'USER_NOM_A', 'USER_EDIFI', 'Num_Planta']].head(10))

  USER_Espai USER_NOM_A                      USER_EDIFI  Num_Planta
0     K/0037   AP-FTI02  FAC. TRADUCCIÓ I INTERPRETACIÓ           0
1     K/2051   AP-FTI05  FAC. TRADUCCIÓ I INTERPRETACIÓ           2
2     K/2027   AP-FTI06  FAC. TRADUCCIÓ I INTERPRETACIÓ           2
3     K/1021   AP-FTI07  FAC. TRADUCCIÓ I INTERPRETACIÓ           1
4     K/2014   AP-FTI08  FAC. TRADUCCIÓ I INTERPRETACIÓ           2
5     K/1064   AP-FTI09  FAC. TRADUCCIÓ I INTERPRETACIÓ           1
6     K/1061   AP-FTI10  FAC. TRADUCCIÓ I INTERPRETACIÓ           1
7     K/1053   AP-FTI11  FAC. TRADUCCIÓ I INTERPRETACIÓ           1
8     K/1038   AP-FTI12  FAC. TRADUCCIÓ I INTERPRETACIÓ           1
9     K/0060   AP-FTI13  FAC. TRADUCCIÓ I INTERPRETACIÓ           0


In [20]:
geo[geo['USER_EDIFI']=='ETSE']

,USER_Espai,X,Y,Nom_Edific,Num_Planta,Ref_Curta,USER_EDIFI,USER_NOM_A,USER_PLANT,geometry
716,QC/0107,425945.112802,4.594597e+06,Q,0,"QC/0107, Q, 0",ETSE,AP-QUIMETS04,0,POINT (2.11273 41.49957)
718,QC/0067,425918.318329,4.594621e+06,Q,0,"QC/0067, Q, 0",ETSE,AP-QUIMETS06,0,POINT (2.11241 41.49978)
719,QC/0081,425928.220395,4.594612e+06,Q,0,"QC/0081, Q, 0",ETSE,AP-QUIMETS05,0,POINT (2.11253 41.49971)
720,QP/0005,425899.861780,4.594595e+06,Q,0,"QP/0005, Q, 0",ETSE,AP-QUIMETS08,0,POINT (2.11219 41.49956)
721,QP/0008,425912.456086,4.594587e+06,Q,0,"QP/0008, Q, 0",ETSE,AP-QUIMETS07,0,POINT (2.11234 41.49948)
...,...,...,...,...,...,...,...,...,...,...
797,QC/3093A,425916.247462,4.594623e+06,Q,3,"QC/3093A, Q, 3",ETSE,AP-ETSE70,3,POINT (2.11238 41.4998)
798,Q2/1003,425909.583083,4.594686e+06,Q,1,"Q2/1003, Q, 1",ETSE,AP-ETSE75,1,POINT (2.11229 41.50037)
799,QC/0001,425859.825489,4.594668e+06,Q,0,"QC/0001, Q, 0",ETSE,AP-ETSE77,0,POINT (2.1117 41.50021)
801,QC/1074,425825.629550,4.594697e+06,Q,1,"QC/1074, Q, 1",ETSE,AP-ETSE78,1,POINT (2.11129 41.50046)


Dictionary to lookup.

In [12]:
room_lookup = {}
for _, row in geo.iterrows():
    room_code = str(row['USER_Espai']).strip().upper()
    room_lookup[room_code] = {
        'ap_name':  str(row['USER_NOM_A']).strip().upper(),
        'building': row['USER_EDIFI'],
        'floor':    row['Num_Planta']
    }

Query

In [13]:
def get_ap(room_code):
    return room_lookup.get(room_code.strip().upper())

In [21]:
print(get_ap('Q1/0007'))

{'ap_name': 'AP-ETSE80', 'building': 'ETSE', 'floor': 0}


Okey, now we have a way to obtain the access point per room. We will start with saving the bookings, it will be of this format:

        {
            'booking_id':  'A3F7B2', #THE TEACHER DOESN'T SEE IT
            'teacher_id':  'ivetsanchez@uab.cat',
            'room_code':   'Q1/0007',
            'ap_name':     'AP-FTI02', #THE TEACHER DOESN'T SEE IT
            'date':        '2025-05-01',
            'start_hour':  10,
            'end_hour':    12,
            'n_students':  30,
            'performance': 'Good'
        }

To generate the booking ID we will use [uuid.uuid4()](https://docs.python.org/3/library/uuid.html) library.

In [34]:
import uuid
import streamlit as st
from datetime import datetime


In [33]:
PERF_RANK = {'Critical':0, 'Poor':1, 'Fair':2, 'Good':3, 
             'Excellent':4, 'Excellent+':5, 'Excellent++':6}

In [24]:
if 'bookings' not in st.session_state:
    st.session_state.bookings = [] 

2026-05-28 09:30:37.435 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-28 09:30:37.436 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-05-28 09:30:37.436 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-28 09:30:37.440 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [25]:
def create_booking(teacher_id, room_code, date, start_hour, end_hour, n_students, min_performance):
    ap_info = get_ap(room_code)
    
    booking = {
        'booking_id':    str(uuid.uuid4())[:8].upper(),
        'teacher_id':    teacher_id,
        'room_code':     room_code,
        'date':          date,
        'start_hour':    start_hour,
        'end_hour':      end_hour,
        'n_students':    n_students,
        'min_performance': min_performance,
        '_ap_name':      ap_info['ap_name'],
    }
    
    st.session_state.bookings.append(booking)
    return booking


In [27]:
def check_availability(bookings, room_code, date, start_hour, end_hour):
    ap_info = get_ap(room_code)
    if ap_info is None:
        return False, None
    ap_name = ap_info['ap_name']
    

    for book in bookings:
        if book['ap_name'] != ap_name or book['date'] != date: #is there something identical?
            continue
        if not (end_hour <= book['start_hour'] or start_hour >= book['end_hour']): #is the thing that is identical between the hours of the book?
            return False, book #no booking available
        
    return True, None #is available

In [30]:
def cancel_booking(bookings, booking_id):
    for i, book in enumerate(bookings):
        if book['booking_id'] == booking_id:
            bookings.pop(i) #we remove it
            return True
        
    return False

In [29]:
def get_bookings_room(bookings, room_code, date):
    return [book for book in bookings if book['room_code']==room_code and book['date']==date]
    
def get_bookings_day(bookings, date):
    return [book for book in bookings if book['date']==date]

def get_bookings_teacher(bookings, teacher_id):
    return [book for book in bookings if book['teacher_id'] ==teacher_id]

In [ ]:
def predict_for_room(room_code, date,start_hour,end_hour, n_students, df_clean):
    # as the prediction only works well till the 5 hours:
    booking_dt = datetime.strptime(f"{date} {start_hour:02d}:00", "%Y-%m-%d %H:%M")
    hours_until = (booking_dt - datetime.now()).total_seconds() / 3600

    if hours_until > 5:
        return {
            'ap_name':     get_ap(room_code)['ap_name'],
            'performance': None,
            'predictions': None,
            'warning':     f"Prediction not available — booking is {hours_until:.0f}h away (max 5h)"
        }


    ap_info = get_ap(room_code)
    if ap_info is None:
        return None
    ap_name = ap_info['ap_name']

    day_of_week = pd.Timestamp(date).dayofweek
    hours = list(range(start_hour, end_hour))

    input_df = df_clean[                          
        (df_clean['swarm_name'] == ap_name) &
        (df_clean['hour'].isin(hours)) &
        (df_clean['snapshot_ts'].dt.dayofweek == day_of_week)
    ].copy()

    if len(input_df) == 0:
        return None
        
    input_df['client_count'] = n_students
    input_df['overloaded']   = n_students > 50
    
    result = PREDICTORFUNCTION(input_row)

    worst = min(result, key=lambda x: PERF_RANK.get(x[1], 0))  #pick the worst prediction across all rows, teacher cares about the worst case in the slot.

    return {
        'performance': worst[1], 
        'predictions': result,
        'ap_name':     ap_name,
        'warning':     None
    }